# 04 – Car Parts Dataset Cleaning and Preparation

## Purpose

This notebook cleans, restructures, validates, and prepares the car-parts segmentation dataset for modeling.

## 1. Setup

### 1.1 Install Dependency

In [ ]:
%pip install -q scikit-learn

### 1.2 Import Libraries

In [1]:
from pathlib import Path
from collections import Counter
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.patches import Polygon
from PIL import Image
from sklearn.model_selection import train_test_split

## 2. Configuration and Paths

### 2.1 Dataset Paths and Reproducibility

In [ ]:
DATASET_ROOT = Path("../data/carparts-seg").resolve()
PROCESSED_ROOT = Path("../data/carparts-seg-processed").resolve()
CONFIG_PATH = DATASET_ROOT / "carparts-seg.yaml"

SPLITS = ["train", "val", "test"]
RANDOM_SEED = 42

In [3]:
print("Original dataset:", DATASET_ROOT)
print("Original dataset exists:", DATASET_ROOT.exists())
print("Configuration file exists:", CONFIG_PATH.exists())
print("Processed dataset:", PROCESSED_ROOT)

Original dataset: C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-assessment\data\carparts-seg
Original dataset exists: True
Configuration file exists: True
Processed dataset: C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-assessment\data\carparts-seg-processed


## 3. Data Loading

### 3.1 Load the Original Dataset Configuration

In [4]:
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

class_names = config["names"]

print("Number of classes:", len(class_names))
print("Classes:", class_names)

Number of classes: 23
Classes: {0: 'back_bumper', 1: 'back_door', 2: 'back_glass', 3: 'back_left_door', 4: 'back_left_light', 5: 'back_light', 6: 'back_right_door', 7: 'back_right_light', 8: 'front_bumper', 9: 'front_door', 10: 'front_glass', 11: 'front_left_door', 12: 'front_left_light', 13: 'front_light', 14: 'front_right_door', 15: 'front_right_light', 16: 'hood', 17: 'left_mirror', 18: 'object', 19: 'right_mirror', 20: 'tailgate', 21: 'trunk', 22: 'wheel'}


### 3.2 Count Original Image and Label Files

In [5]:
baseline_records = []

for split in SPLITS:
    image_dir = DATASET_ROOT / "images" / split
    label_dir = DATASET_ROOT / "labels" / split

    image_count = len(list(image_dir.glob("*.jpg")))
    label_count = len(list(label_dir.glob("*.txt")))

    baseline_records.append({
        "Split": split,
        "Images": image_count,
        "Label Files": label_count
    })

baseline_df = pd.DataFrame(baseline_records)
baseline_df

,Split,Images,Label Files
0,train,3156,3156
1,val,401,401
2,test,276,276


## 4. Data Validation

### 4.1 Identify Empty Label Files

In [6]:
empty_label_files = []

for split in SPLITS:
    label_dir = DATASET_ROOT / "labels" / split

    for label_path in label_dir.glob("*.txt"):
        content = label_path.read_text(encoding="utf-8").strip()

        if not content:
            empty_label_files.append(label_path)
            
print("Empty label files:", len(empty_label_files))

Empty label files: 135


In [7]:
empty_labels_by_split = Counter(
    label_path.parent.name
    for label_path in empty_label_files
)

empty_labels_df = pd.DataFrame(
    empty_labels_by_split.items(),
    columns=["Split", "Empty Label Files"]
)

empty_labels_df

,Split,Empty Label Files
0,train,116
1,val,12
2,test,7


In [8]:
empty_label_records = []

for label_path in empty_label_files:
    split = label_path.parent.name
    image_path = DATASET_ROOT / "images" / split / f"{label_path.stem}.jpg"

    empty_label_records.append({
        "Split": split,
        "Image Path": image_path,
        "Label Path": label_path,
        "Image Exists": image_path.exists()
    })

empty_labels_review_df = pd.DataFrame(empty_label_records)

empty_labels_review_df.head()

,Split,Image Path,Label Path,Image Exists
0,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
1,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
2,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
3,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
4,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True


In [9]:
print("Images found:", empty_labels_review_df["Image Exists"].sum())
print("Images missing:", (~empty_labels_review_df["Image Exists"]).sum())

Images found: 135
Images missing: 0


### 4.2 Identify the Unclear Object Class

In [10]:
OBJECT_CLASS_ID = 18
object_label_files = []

for split in SPLITS:
    label_dir = DATASET_ROOT / "labels" / split

    for label_path in label_dir.glob("*.txt"):
        lines = label_path.read_text(encoding="utf-8").splitlines()

        for line in lines:
            if not line.strip():
                continue

            class_id = int(line.split()[0])

            if class_id == OBJECT_CLASS_ID:
                object_label_files.append(label_path)
                break

In [11]:
print("Images containing the object class:", len(object_label_files))

Images containing the object class: 12


In [12]:
object_records = []

for label_path in object_label_files:
    split = label_path.parent.name
    image_path = DATASET_ROOT / "images" / split / f"{label_path.stem}.jpg"

    object_records.append({
        "Split": split,
        "Image Path": image_path,
        "Label Path": label_path,
        "Image Exists": image_path.exists()
    })

object_review_df = pd.DataFrame(object_records)
object_review_df

,Split,Image Path,Label Path,Image Exists
0,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
1,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
2,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
3,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
4,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
5,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
6,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
7,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
8,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True
9,train,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-a...,True


### 4.3 Review Partial Annotations

In [13]:
random.seed(RANDOM_SEED)

PARTIAL_REVIEW_SAMPLE_SIZE = 30
partial_review_records = []

In [14]:
excluded_labels = set(empty_label_files + object_label_files)

for split in SPLITS:
    label_dir = DATASET_ROOT / "labels" / split

    eligible_labels = [
        label_path
        for label_path in label_dir.glob("*.txt")
        if label_path not in excluded_labels
    ]

    sampled_labels = random.sample(
        eligible_labels,
        k=min(PARTIAL_REVIEW_SAMPLE_SIZE, len(eligible_labels))
    )

    for label_path in sampled_labels:
        image_path = (
            DATASET_ROOT
            / "images"
            / split
            / f"{label_path.stem}.jpg"
        )

        partial_review_records.append({
            "Split": split,
            "Image Path": image_path,
            "Label Path": label_path
        })

In [15]:
partial_review_df = pd.DataFrame(partial_review_records)

partial_review_df.groupby("Split").size()

Split
test     30
train    30
val      30
dtype: int64

### 4.4 Prepare the Manual Review Package

In [16]:
ROBOFLOW_REVIEW_DIR = Path(
    "../data/review/roboflow-review-package"
).resolve()

if ROBOFLOW_REVIEW_DIR.exists():
    shutil.rmtree(ROBOFLOW_REVIEW_DIR)

ROBOFLOW_IMAGES_DIR = ROBOFLOW_REVIEW_DIR / "images" / "train"
ROBOFLOW_LABELS_DIR = ROBOFLOW_REVIEW_DIR / "labels" / "train"

ROBOFLOW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
ROBOFLOW_LABELS_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
review_manifest = []

def add_to_review_package(image_path, label_path, split, issue):
    review_name = f"{split}__{image_path.name}"

    image_destination = ROBOFLOW_IMAGES_DIR / review_name
    label_destination = (
        ROBOFLOW_LABELS_DIR
        / f"{Path(review_name).stem}.txt"
    )

    shutil.copy2(image_path, image_destination)

    if label_path.exists() and label_path.stat().st_size > 0:
        shutil.copy2(label_path, label_destination)

    review_manifest.append({
        "Review Name": review_name,
        "Original Split": split,
        "Original Image": str(image_path),
        "Original Label": str(label_path),
        "Issue": issue
    })

In [18]:
for record in object_records:
    add_to_review_package(
        image_path=record["Image Path"],
        label_path=record["Label Path"],
        split=record["Split"],
        issue="object_class"
    )

In [19]:
for record in partial_review_records:
    add_to_review_package(
        image_path=record["Image Path"],
        label_path=record["Label Path"],
        split=record["Split"],
        issue="partial_annotation_review"
    )

In [20]:
review_manifest_df = pd.DataFrame(review_manifest)

review_manifest_df.to_csv(
    ROBOFLOW_REVIEW_DIR / "review_manifest.csv",
    index=False
)

review_manifest_df.groupby("Issue").size()

Issue
object_class                 12
partial_annotation_review    90
dtype: int64

In [21]:
review_images = list(
    (ROBOFLOW_REVIEW_DIR / "images").rglob("*.jpg")
)

review_labels = list(
    (ROBOFLOW_REVIEW_DIR / "labels").rglob("*.txt")
)

print("Review images:", len(review_images))
print("Existing label files:", len(review_labels))
print("Manifest records:", len(review_manifest_df))

Review images: 102
Existing label files: 102
Manifest records: 102


In [22]:
review_config = {
    "path": ".",
    "train": "images/train",
    "names": class_names
}

with (ROBOFLOW_REVIEW_DIR / "data.yaml").open(
    "w",
    encoding="utf-8"
) as file:
    yaml.safe_dump(
        review_config,
        file,
        sort_keys=False,
        allow_unicode=True
    )

In [23]:
print(
    (ROBOFLOW_REVIEW_DIR / "data.yaml").read_text(
        encoding="utf-8"
    )
)

path: .
train: images/train
names:
  0: back_bumper
  1: back_door
  2: back_glass
  3: back_left_door
  4: back_left_light
  5: back_light
  6: back_right_door
  7: back_right_light
  8: front_bumper
  9: front_door
  10: front_glass
  11: front_left_door
  12: front_left_light
  13: front_light
  14: front_right_door
  15: front_right_light
  16: hood
  17: left_mirror
  18: object
  19: right_mirror
  20: tailgate
  21: trunk
  22: wheel



In [24]:
duplicate_review_names = review_manifest_df[
    "Review Name"
].duplicated().sum()

print("Duplicate image names:", duplicate_review_names)

Duplicate image names: 0


### 4.5 Record Manual Review Results

In [25]:
review_results = {
    "wheel_missing": 55,
    "wheel_partial": 0,
    "wheel_not_visible": 31,
    "wheel_complete": 14,
    "other_parts_partial": 48,
    "invalid": 2
}

review_results_df = pd.DataFrame(
    review_results.items(),
    columns=["Issue", "Images"]
)

review_results_df["Percentage"] = (
    review_results_df["Images"]
    / len(review_manifest_df)
    * 100
).round(2)

review_results_df

,Issue,Images,Percentage
0,wheel_missing,55,53.92
1,wheel_partial,0,0.00
2,wheel_not_visible,31,30.39
3,wheel_complete,14,13.73
4,other_parts_partial,48,47.06
5,invalid,2,1.96


In [28]:
visible_wheel_images = (
    review_results["wheel_missing"]
    + review_results["wheel_partial"]
    + review_results["wheel_complete"]
)

incomplete_wheel_images = (
    review_results["wheel_missing"]
    + review_results["wheel_partial"]
)

incomplete_wheel_percentage = (
    incomplete_wheel_images
    / visible_wheel_images
    * 100
)

print("Visible-wheel images:", visible_wheel_images)
print(
    "Incomplete wheel annotation percentage:",
    round(incomplete_wheel_percentage, 2)
)

Visible-wheel images: 69
Incomplete wheel annotation percentage: 79.71


In [27]:
annotation_quality_df = pd.DataFrame({
    "Status": [
        "Partial Annotation",
        "Clean",
        "Invalid"
    ],
    "Images": [70, 30, 2]
})

annotation_quality_df["Percentage"] = (
    annotation_quality_df["Images"]
    / len(review_manifest_df)
    * 100
).round(2)

annotation_quality_df

,Status,Images,Percentage
0,Partial Annotation,70,68.63
1,Clean,30,29.41
2,Invalid,2,1.96


### 4.6 Validation Decisions

- Exclude the 135 empty-label files and the 12 files containing the unclear `object` class.
- Retain partially annotated images for the baseline, but do not deliberately target them for augmentation.
- Merge direction-specific doors, lights, and mirrors into general part classes.
- Rebuild the split at source-image level so augmented versions of one source cannot cross splits.

## 5. Analysis and Visualization

### 5.1 Original Training Class Distribution

In [30]:
train_instance_counts = Counter()
train_image_counts = Counter()

train_label_dir = DATASET_ROOT / "labels" / "train"

for label_path in train_label_dir.glob("*.txt"):
    lines = label_path.read_text(
        encoding="utf-8"
    ).splitlines()

    image_class_ids = set()

    for line in lines:
        if not line.strip():
            continue

        class_id = int(line.split()[0])

        train_instance_counts[class_id] += 1
        image_class_ids.add(class_id)

    for class_id in image_class_ids:
        train_image_counts[class_id] += 1

In [31]:
class_distribution = []

for class_id, class_name in class_names.items():
    class_distribution.append({
        "Class ID": class_id,
        "Class": class_name,
        "Instances": train_instance_counts[class_id],
        "Images": train_image_counts[class_id]
    })

class_distribution_df = pd.DataFrame(class_distribution)

class_distribution_df = class_distribution_df.sort_values(
    "Images"
).reset_index(drop=True)

class_distribution_df

,Class ID,Class,Instances,Images
0,18,object,10,10
1,20,tailgate,88,88
2,21,trunk,146,146
3,6,back_right_door,184,184
4,7,back_right_light,204,204
5,3,back_left_door,204,204
6,14,front_right_door,208,208
7,11,front_left_door,224,224
8,4,back_left_light,244,244
9,15,front_right_light,392,392


### 5.2 Merged Class Distribution

In [32]:
class_merge_map = {
    "front_left_door": "front_door",
    "front_right_door": "front_door",
    "back_left_door": "back_door",
    "back_right_door": "back_door",
    "front_left_light": "front_light",
    "front_right_light": "front_light",
    "back_left_light": "back_light",
    "back_right_light": "back_light",
    "left_mirror": "side_mirror",
    "right_mirror": "side_mirror"
}

final_class_names = [
    "back_bumper",
    "back_door",
    "back_glass",
    "back_light",
    "front_bumper",
    "front_door",
    "front_glass",
    "front_light",
    "hood",
    "side_mirror",
    "tailgate",
    "trunk",
    "wheel"
]

In [33]:
merged_instance_counts = Counter()
merged_image_counts = Counter()

for label_path in train_label_dir.glob("*.txt"):
    lines = label_path.read_text(
        encoding="utf-8"
    ).splitlines()

    image_classes = set()

    for line in lines:
        if not line.strip():
            continue

        class_id = int(line.split()[0])
        original_name = class_names[class_id]

        if original_name == "object":
            continue

        merged_name = class_merge_map.get(
            original_name,
            original_name
        )

        merged_instance_counts[merged_name] += 1
        image_classes.add(merged_name)

    for class_name in image_classes:
        merged_image_counts[class_name] += 1

In [34]:
merged_distribution_df = pd.DataFrame({
    "Class": final_class_names,
    "Instances": [
        merged_instance_counts[name]
        for name in final_class_names
    ],
    "Images": [
        merged_image_counts[name]
        for name in final_class_names
    ]
})

merged_distribution_df = merged_distribution_df.sort_values(
    "Images"
).reset_index(drop=True)

merged_distribution_df

,Class,Instances,Images
0,tailgate,88,88
1,trunk,146,146
2,wheel,709,492
3,side_mirror,816,584
4,back_bumper,732,732
5,back_glass,890,886
6,back_door,1396,1392
7,back_light,2036,1426
8,front_door,1582,1580
9,front_bumper,1616,1616


### 5.3 Original Cross-Split Leakage

In [35]:
def get_source_name(image_path):
    return image_path.stem.split(".rf.")[0]


image_records = []

for split in SPLITS:
    image_dir = DATASET_ROOT / "images" / split

    for image_path in image_dir.glob("*.jpg"):
        image_records.append({
            "Split": split,
            "Image Name": image_path.name,
            "Source Name": get_source_name(image_path)
        })

image_sources_df = pd.DataFrame(image_records)

image_sources_df.head()

,Split,Image Name,Source Name
0,train,car10_jpg.rf.01c1346d0a306c6ba1a1b9c3fa7c9599.jpg,car10_jpg
1,train,car10_jpg.rf.291cfe46c25830e83dd4283cb304d159.jpg,car10_jpg
2,train,car10_jpg.rf.2f776f55ff2ce1799469b62c70ab650f.jpg,car10_jpg
3,train,car10_jpg.rf.721a1794b2f63ec462c53e604b9657f3.jpg,car10_jpg
4,train,car10_jpg.rf.9ed8f650cba0a1dfeaa579453dc65608.jpg,car10_jpg


In [36]:
source_split_counts = (
    image_sources_df
    .groupby("Source Name")["Split"]
    .nunique()
)

cross_split_source_names = source_split_counts[
    source_split_counts > 1
].index

cross_split_duplicates_df = (
    image_sources_df[
        image_sources_df["Source Name"].isin(
            cross_split_source_names
        )
    ]
    .sort_values(["Source Name", "Split"])
)

print(
    "Sources appearing in multiple splits:",
    len(cross_split_source_names)
)

print(
    "Image files involved:",
    len(cross_split_duplicates_df)
)

cross_split_duplicates_df.head(20)

Sources appearing in multiple splits: 429
Image files involved: 2647


,Split,Image Name,Source Name
8,train,car118_jpg.rf.35dcf39fa45e5c008abd850acfaf3f4c...,car118_jpg
9,train,car118_jpg.rf.42387f2f4da28f2cb336f08ffde3f1ec...,car118_jpg
10,train,car118_jpg.rf.710deebdd6995e0c8dbdf4f01fd0f077...,car118_jpg
11,train,car118_jpg.rf.75a2b0bb352b3a9002a612fdce15cfd8...,car118_jpg
12,train,car118_jpg.rf.cb73c4c418e3fdc2ddd664a8b4530d10...,car118_jpg
13,train,car118_jpg.rf.d210a3fa11e32c77a7999140c4d60a12...,car118_jpg
3156,val,car118_jpg.rf.1c585f41d29960bafd3da69fc63600ea...,car118_jpg
3557,test,car25_jpg.rf.a2a62c4f3c8af7837e5ee62d1b806750.jpg,car25_jpg
38,train,car25_jpg.rf.40c110c727224035bc9e612d9c604d82.jpg,car25_jpg
39,train,car25_jpg.rf.5266f1de032f4802d7e613f11c3e4f31.jpg,car25_jpg


In [37]:
files_per_source = (
    image_sources_df
    .groupby("Source Name")
    .size()
)

print("Total image files:", len(image_sources_df))
print("Unique source images:", image_sources_df["Source Name"].nunique())
print("Sources with multiple versions:", (files_per_source > 1).sum())

files_per_source.describe()

Total image files: 3833
Unique source images: 585
Sources with multiple versions: 585


count    585.000000
mean       6.552137
std        1.100015
min        3.000000
25%        6.000000
50%        7.000000
75%        7.000000
max        8.000000
dtype: float64

In [38]:
split_combinations = (
    cross_split_duplicates_df
    .groupby("Source Name")["Split"]
    .apply(lambda splits: " + ".join(sorted(set(splits))))
    .value_counts()
)

split_combinations

Split
train + val           201
test + train          134
test + train + val     89
test + val              5
Name: count, dtype: int64

### 5.4 Source-Level Class Analysis

In [39]:
source_class_records = []

for record in image_sources_df.to_dict("records"):
    label_path = (
        DATASET_ROOT
        / "labels"
        / record["Split"]
        / Path(record["Image Name"]).with_suffix(".txt").name
    )

    if not label_path.exists():
        continue

    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        class_id = int(line.split()[0])
        class_name = class_names[class_id]

        if class_name == "object":
            continue

        merged_class = class_merge_map.get(
            class_name,
            class_name
        )

        source_class_records.append({
            "Source Name": record["Source Name"],
            "Class": merged_class
        })

source_classes_df = (
    pd.DataFrame(source_class_records)
    .drop_duplicates(["Source Name", "Class"])
)

source_class_distribution_df = (
    source_classes_df
    .groupby("Class")["Source Name"]
    .nunique()
    .sort_values()
    .reset_index(name="Unique Source Images")
)

source_class_distribution_df

,Class,Unique Source Images
0,tailgate,57
1,trunk,100
2,back_bumper,154
3,back_glass,168
4,back_light,291
5,front_glass,332
6,hood,339
7,front_bumper,353
8,wheel,355
9,back_door,357


## 6. Processing

### 6.1 Create the Balanced Source-Level Split

In [52]:
source_class_matrix = pd.crosstab(
    source_classes_df["Source Name"],
    source_classes_df["Class"]
).clip(upper=1)


def create_source_split(random_state):
    sources = source_class_matrix.index.to_list()

    train, remaining = train_test_split(
        sources,
        test_size=0.30,
        random_state=random_state
    )

    val, test = train_test_split(
        remaining,
        test_size=0.50,
        random_state=random_state
    )

    return {
        "train": train,
        "val": val,
        "test": test
    }


def calculate_balance_score(split_sources):
    target_ratios = {
        "train": 0.70,
        "val": 0.15,
        "test": 0.15
    }

    total_per_class = source_class_matrix.sum()
    differences = []

    for split, sources in split_sources.items():
        split_counts = source_class_matrix.loc[sources].sum()
        split_ratios = split_counts / total_per_class

        differences.extend(
            abs(split_ratios - target_ratios[split])
        )

    return np.mean(differences)

In [53]:
split_scores = {
    random_state: calculate_balance_score(
        create_source_split(random_state)
    )
    for random_state in range(1000)
}

best_random_state = min(
    split_scores,
    key=split_scores.get
)

print("Best random state:", best_random_state)
print("Balance score:", split_scores[best_random_state])

Best random state: 557
Balance score: 0.006205895361530172


In [54]:
best_split_sources = create_source_split(
    best_random_state
)

source_split_map = {
    source_name: split
    for split, source_names in best_split_sources.items()
    for source_name in source_names
}

image_sources_df["New Split"] = (
    image_sources_df["Source Name"]
    .map(source_split_map)
)

In [55]:
split_summary_df = pd.DataFrame({
    "Unique Sources": (
        image_sources_df
        .groupby("New Split")["Source Name"]
        .nunique()
    ),
    "Image Files": (
        image_sources_df
        .groupby("New Split")
        .size()
    )
}).reindex(SPLITS)

split_summary_df

,Unique Sources,Image Files
New Split,,
train,409,2675
val,88,579
test,88,579


In [56]:
source_assignments_df = pd.DataFrame(
    source_split_map.items(),
    columns=["Source Name", "New Split"]
)

source_classes_with_split_df = source_classes_df.merge(
    source_assignments_df,
    on="Source Name",
    how="left"
)

class_split_distribution_df = pd.crosstab(
    source_classes_with_split_df["Class"],
    source_classes_with_split_df["New Split"]
)

class_split_distribution_df = (
    class_split_distribution_df
    .reindex(columns=SPLITS, fill_value=0)
    .rename(columns={
        "train": "Train Sources",
        "val": "Val Sources",
        "test": "Test Sources"
    })
)

class_split_distribution_df["Total Sources"] = (
    class_split_distribution_df.sum(axis=1)
)

class_split_distribution_df.sort_values(
    "Total Sources"
)

New Split,Train Sources,Val Sources,Test Sources,Total Sources
Class,,,,
tailgate,39,10,8,57
trunk,70,15,15,100
back_bumper,110,22,22,154
back_glass,120,23,25,168
back_light,202,48,41,291
front_glass,235,48,49,332
hood,239,48,52,339
front_bumper,248,52,53,353
wheel,250,54,51,355


### 6.2 Validate the Source-Level Split

In [57]:
new_source_split_counts = (
    image_sources_df
    .groupby("Source Name")["New Split"]
    .nunique()
)

remaining_leakage = (
    new_source_split_counts > 1
).sum()

print(
    "Sources appearing in multiple new splits:",
    remaining_leakage
)

Sources appearing in multiple new splits: 0


### 6.3 Define the Final Classes

In [58]:
processed_class_names = [
    "back_bumper",
    "back_door",
    "back_glass",
    "back_light",
    "front_bumper",
    "front_door",
    "front_glass",
    "front_light",
    "hood",
    "side_mirror",
    "tailgate",
    "trunk",
    "wheel"
]

processed_class_ids = {
    class_name: class_id
    for class_id, class_name
    in enumerate(processed_class_names)
}

processed_class_ids

{'back_bumper': 0,
 'back_door': 1,
 'back_glass': 2,
 'back_light': 3,
 'front_bumper': 4,
 'front_door': 5,
 'front_glass': 6,
 'front_light': 7,
 'hood': 8,
 'side_mirror': 9,
 'tailgate': 10,
 'trunk': 11,
 'wheel': 12}

### 6.4 Select Files for Processing

In [59]:
processing_records = []

for record in image_sources_df.to_dict("records"):
    label_path = (
        DATASET_ROOT
        / "labels"
        / record["Split"]
        / Path(record["Image Name"]).with_suffix(".txt").name
    )

    if not label_path.exists():
        decision = "exclude_missing_label"

    else:
        annotation_lines = [
            line
            for line in label_path.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        if not annotation_lines:
            decision = "exclude_empty_label"

        else:
            annotation_classes = [
                class_names[int(line.split()[0])]
                for line in annotation_lines
            ]

            if "object" in annotation_classes:
                decision = "exclude_object_class"
            else:
                decision = "keep"

    processing_records.append({
        "Original Split": record["Split"],
        "New Split": record["New Split"],
        "Image Name": record["Image Name"],
        "Source Name": record["Source Name"],
        "Decision": decision
    })

processing_manifest_df = pd.DataFrame(
    processing_records
)

In [60]:
processing_manifest_df[
    "Decision"
].value_counts()

Decision
keep                    3686
exclude_empty_label      135
exclude_object_class      12
Name: count, dtype: int64

### 6.5 Validate Selected Files

In [61]:
kept_manifest_df = processing_manifest_df[
    processing_manifest_df["Decision"] == "keep"
].copy()

processed_split_summary_df = pd.DataFrame({
    "Unique Sources": (
        kept_manifest_df
        .groupby("New Split")["Source Name"]
        .nunique()
    ),
    "Image Files": (
        kept_manifest_df
        .groupby("New Split")
        .size()
    )
}).reindex(SPLITS)

processed_split_summary_df

,Unique Sources,Image Files
New Split,,
train,409,2571
val,88,561
test,88,554


In [62]:
duplicate_output_names = (
    kept_manifest_df
    .duplicated(
        subset=["New Split", "Image Name"]
    )
    .sum()
)

print(
    "Duplicate filenames within new splits:",
    duplicate_output_names
)

print(
    "Selected image files:",
    len(kept_manifest_df)
)

Duplicate filenames within new splits: 0
Selected image files: 3686


### 6.6 Create the Processed Dataset

In [63]:
existing_output_files = []

if PROCESSED_ROOT.exists():
    existing_output_files = [
        path
        for path in PROCESSED_ROOT.rglob("*")
        if path.is_file()
    ]

if existing_output_files:
    raise FileExistsError(
        f"{PROCESSED_ROOT} already contains files."
    )

In [64]:
for split in SPLITS:
    (
        PROCESSED_ROOT / "images" / split
    ).mkdir(parents=True, exist_ok=True)

    (
        PROCESSED_ROOT / "labels" / split
    ).mkdir(parents=True, exist_ok=True)

In [65]:
for record in kept_manifest_df.to_dict("records"):
    original_split = record["Original Split"]
    new_split = record["New Split"]
    image_name = record["Image Name"]
    label_name = Path(image_name).with_suffix(".txt").name

    source_image_path = (
        DATASET_ROOT
        / "images"
        / original_split
        / image_name
    )

    source_label_path = (
        DATASET_ROOT
        / "labels"
        / original_split
        / label_name
    )

    output_image_path = (
        PROCESSED_ROOT
        / "images"
        / new_split
        / image_name
    )

    output_label_path = (
        PROCESSED_ROOT
        / "labels"
        / new_split
        / label_name
    )

    shutil.copy2(
        source_image_path,
        output_image_path
    )

    transformed_lines = []

    for line in source_label_path.read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue

        values = line.split()
        original_class_id = int(values[0])
        original_class_name = class_names[
            original_class_id
        ]

        merged_class_name = class_merge_map.get(
            original_class_name,
            original_class_name
        )

        new_class_id = processed_class_ids[
            merged_class_name
        ]

        transformed_line = " ".join([
            str(new_class_id),
            *values[1:]
        ])

        transformed_lines.append(
            transformed_line
        )

    output_label_path.write_text(
        "\n".join(transformed_lines) + "\n",
        encoding="utf-8"
    )

In [66]:
kept_manifest_df.to_csv(
    PROCESSED_ROOT / "processing_manifest.csv",
    index=False
)

print("Processed files created:", len(kept_manifest_df))

Processed files created: 3686


### 6.7 Create the Processed Dataset YAML

In [67]:
processed_config = {
    "path": PROCESSED_ROOT.as_posix(),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        class_id: class_name
        for class_id, class_name
        in enumerate(processed_class_names)
    }
}

processed_config_path = (
    PROCESSED_ROOT
    / "carparts-seg-processed.yaml"
)

with processed_config_path.open(
    "w",
    encoding="utf-8"
) as yaml_file:
    yaml.safe_dump(
        processed_config,
        yaml_file,
        sort_keys=False,
        allow_unicode=True
    )

print("YAML created at:")
print(processed_config_path)

YAML created at:
C:\Users\Hai-1\Desktop\qaddir-vehicle-damage-assessment\data\carparts-seg-processed\carparts-seg-processed.yaml


In [68]:
with processed_config_path.open(
    "r",
    encoding="utf-8"
) as yaml_file:
    saved_processed_config = yaml.safe_load(
        yaml_file
    )

saved_processed_config

{'path': 'C:/Users/Hai-1/Desktop/qaddir-vehicle-damage-assessment/data/carparts-seg-processed',
 'train': 'images/train',
 'val': 'images/val',
 'test': 'images/test',
 'names': {0: 'back_bumper',
  1: 'back_door',
  2: 'back_glass',
  3: 'back_light',
  4: 'front_bumper',
  5: 'front_door',
  6: 'front_glass',
  7: 'front_light',
  8: 'hood',
  9: 'side_mirror',
  10: 'tailgate',
  11: 'trunk',
  12: 'wheel'}}

## 7. Results

### 7.1 Validate Image and Label Pairs

In [69]:
processed_file_validation = []

for split in SPLITS:
    image_dir = (
        PROCESSED_ROOT
        / "images"
        / split
    )

    label_dir = (
        PROCESSED_ROOT
        / "labels"
        / split
    )

    image_files = list(
        image_dir.glob("*.jpg")
    )

    label_files = list(
        label_dir.glob("*.txt")
    )

    image_stems = {
        path.stem
        for path in image_files
    }

    label_stems = {
        path.stem
        for path in label_files
    }

    processed_file_validation.append({
        "Split": split,
        "Images": len(image_files),
        "Labels": len(label_files),
        "Images Without Labels": len(
            image_stems - label_stems
        ),
        "Labels Without Images": len(
            label_stems - image_stems
        )
    })

processed_file_validation_df = pd.DataFrame(
    processed_file_validation
)

processed_file_validation_df

,Split,Images,Labels,Images Without Labels,Labels Without Images
0,train,2571,2571,0,0
1,val,561,561,0,0
2,test,554,554,0,0


In [70]:
print(
    "Total processed images:",
    processed_file_validation_df["Images"].sum()
)

print(
    "Total processed labels:",
    processed_file_validation_df["Labels"].sum()
)

Total processed images: 3686
Total processed labels: 3686


### 7.2 Validate Segmentation Labels

In [ ]:
annotation_validation = []
annotation_errors = []

for split in SPLITS:
    label_files = sorted(
        (PROCESSED_ROOT / "labels" / split).glob("*.txt")
    )

    empty_labels = 0
    invalid_class_lines = 0
    invalid_polygon_lines = 0
    out_of_range_lines = 0
    polygon_instances = 0

    for label_path in label_files:
        content = label_path.read_text(encoding="utf-8").strip()

        if not content:
            empty_labels += 1
            annotation_errors.append((split, label_path.name, "empty label"))
            continue

        for line_number, line in enumerate(content.splitlines(), start=1):
            tokens = line.split()

            try:
                class_id = int(tokens[0])
            except (IndexError, ValueError):
                invalid_polygon_lines += 1
                annotation_errors.append((split, label_path.name, line_number))
                continue

            if class_id not in range(len(processed_class_names)):
                invalid_class_lines += 1

            coordinate_tokens = tokens[1:]
            structurally_valid = (
                len(coordinate_tokens) >= 6
                and len(coordinate_tokens) % 2 == 0
            )

            if not structurally_valid:
                invalid_polygon_lines += 1
                annotation_errors.append((split, label_path.name, line_number))
                continue

            try:
                coordinates = [float(value) for value in coordinate_tokens]
            except ValueError:
                invalid_polygon_lines += 1
                annotation_errors.append((split, label_path.name, line_number))
                continue

            if any(value < 0.0 or value > 1.0 for value in coordinates):
                out_of_range_lines += 1

            polygon_instances += 1

    annotation_validation.append({
        "Split": split,
        "Empty Labels": empty_labels,
        "Invalid Class Lines": invalid_class_lines,
        "Invalid Polygon Lines": invalid_polygon_lines,
        "Out-of-Range Lines": out_of_range_lines,
        "Polygon Instances": polygon_instances
    })

annotation_validation_df = pd.DataFrame(annotation_validation)
annotation_validation_df

### 7.3 Confirm Source Isolation

In [ ]:
processed_source_records = []

for split in SPLITS:
    for image_path in (PROCESSED_ROOT / "images" / split).glob("*.jpg"):
        processed_source_records.append({
            "Split": split,
            "Source Name": get_source_name(image_path)
        })

processed_sources_df = pd.DataFrame(processed_source_records)
processed_source_split_counts = (
    processed_sources_df
    .groupby("Source Name")["Split"]
    .nunique()
)
processed_leakage_count = int(
    (processed_source_split_counts > 1).sum()
)

print(
    "Sources appearing in multiple processed splits:",
    processed_leakage_count
)

### 7.4 Display Processed Samples

In [ ]:
sample_rng = random.Random(RANDOM_SEED)
sample_records = []

for split in SPLITS:
    split_labels = sorted(
        (PROCESSED_ROOT / "labels" / split).glob("*.txt")
    )
    for label_path in sample_rng.sample(split_labels, k=2):
        sample_records.append((split, label_path))

figure, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = plt.get_cmap("tab20")

for axis, (split, label_path) in zip(axes.flat, sample_records):
    image_path = PROCESSED_ROOT / "images" / split / f"{label_path.stem}.jpg"
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    axis.imshow(image)

    for line in label_path.read_text(encoding="utf-8").splitlines():
        values = line.split()
        class_id = int(values[0])
        coordinates = np.asarray(values[1:], dtype=float).reshape(-1, 2)
        coordinates[:, 0] *= width
        coordinates[:, 1] *= height
        color = colors(class_id / len(processed_class_names))
        axis.add_patch(Polygon(
            coordinates,
            closed=True,
            fill=False,
            edgecolor=color,
            linewidth=2
        ))
        label_position = coordinates.mean(axis=0)
        axis.text(
            label_position[0],
            label_position[1],
            processed_class_names[class_id],
            color="white",
            fontsize=8,
            bbox={"facecolor": color, "alpha": 0.75, "pad": 2}
        )

    axis.set_title(f"{split}: {image_path.name}")
    axis.axis("off")

plt.tight_layout()
plt.show()

### 7.5 Summarize the Processed Dataset

In [ ]:
processed_summary_df = (
    processed_file_validation_df
    .merge(
        processed_sources_df
        .groupby("Split")["Source Name"]
        .nunique()
        .rename("Unique Sources")
        .reset_index(),
        on="Split"
    )
    .merge(
        annotation_validation_df[["Split", "Polygon Instances"]],
        on="Split"
    )
)

processed_summary_df = processed_summary_df[[
    "Split",
    "Unique Sources",
    "Images",
    "Labels",
    "Polygon Instances"
]]

validation_errors = []

if not (processed_summary_df["Images"] == processed_summary_df["Labels"]).all():
    validation_errors.append("Image and label counts do not match.")

if processed_summary_df["Images"].sum() != len(kept_manifest_df):
    validation_errors.append("Processed image count does not match the manifest.")

if annotation_validation_df["Empty Labels"].sum() != 0:
    validation_errors.append("Empty processed label files were found.")

if annotation_validation_df["Invalid Class Lines"].sum() != 0:
    validation_errors.append("Invalid class IDs were found.")

if annotation_validation_df["Invalid Polygon Lines"].sum() != 0:
    validation_errors.append("Invalid polygon structures were found.")

if annotation_validation_df["Out-of-Range Lines"].sum() != 0:
    validation_errors.append("Out-of-range polygon coordinates were found.")

if processed_leakage_count != 0:
    validation_errors.append("Source-level leakage was found.")

if validation_errors:
    error_message = "\n".join(validation_errors)
    raise ValueError(
        f"Processed dataset validation failed:\n{error_message}"
    )

print("Processed dataset validation passed.")

total_row = pd.DataFrame([{
    "Split": "Total",
    "Unique Sources": processed_summary_df["Unique Sources"].sum(),
    "Images": processed_summary_df["Images"].sum(),
    "Labels": processed_summary_df["Labels"].sum(),
    "Polygon Instances": processed_summary_df["Polygon Instances"].sum()
}])

processed_summary_df = pd.concat(
    [processed_summary_df, total_row],
    ignore_index=True
)

processed_summary_df

## 8. Decisions and Next Steps

### 8.1 Compare Baseline and Future Augmentation Settings

In [71]:
baseline_augmentation = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.0,
    "flipud": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0
}

In [72]:
training_augmentation = {
    "hsv_h": 0.01,
    "hsv_s": 0.3,
    "hsv_v": 0.3,
    "degrees": 10.0,
    "translate": 0.1,
    "scale": 0.2,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "bgr": 0.0,
    "mosaic": 0.3,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0
}

In [73]:
augmentation_comparison_df = pd.DataFrame({
    "Baseline": baseline_augmentation,
    "Augmented Experiment": training_augmentation
})

augmentation_comparison_df

,Baseline,Augmented Experiment
hsv_h,0.0,0.01
hsv_s,0.0,0.30
hsv_v,0.0,0.30
degrees,0.0,10.00
translate,0.0,0.10
scale,0.0,0.20
fliplr,0.0,0.50
flipud,0.0,0.00
mosaic,0.0,0.30
mixup,0.0,0.00


### 8.2 Final Decisions

- The original 3,833 files came from 585 independent source images.
- The final source-level split contains 409 train, 88 validation, and 88 test sources with zero leakage.
- Excluding 135 empty-label files and 12 `object`-class files produced 3,686 processed image-label pairs.
- The 23 original classes were merged into 13 final car-part classes.
- Existing augmented files remain in the processed dataset, so the baseline adds no training-time augmentation.
- Validation and test remain untouched evaluation sets; only the training split is eligible for future augmentation.
- The augmented experiment is deferred until the baseline and external-data audit are reviewed.